In [1]:
!pip install plotly

import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display


In [2]:
import pandas as pd

# Fájl neve
filename = "adatok_meteorologia.csv"

# Beolvasás – fejléc a 6. sorban (0-index szerint header=5)
df = pd.read_csv(
    filename,
    sep=";",
    header=5,
    encoding="cp1250",   # ← javított kódolás
    engine="python"
)

# Oszlopnevek tisztítása
df.columns = [c.strip() for c in df.columns]

df.head()


,StationNumber,Time,rau,Q_rau,t,Q_t,tn,Q_tn,tx,Q_tx,...,Q_et20,et50,Q_et50,et100,Q_et100,tsn24,Q_tsn24,tviz,Q_tviz,EOR
0,13711,20031108,-999.0,,5.8,,-999.0,,-999.0,,...,,8.3,,10.8,,-999.0,,-999,,EOR
1,13711,20031109,-999.0,,-999.0,,-999.0,,-999.0,,...,,8.2,,10.8,,-999.0,,-999,,EOR
2,13711,20031110,0.2,,4.9,,-999.0,,-999.0,,...,,8.0,,10.6,,-999.0,,-999,,EOR
3,13711,20031111,-999.0,,3.1,,-0.8,,6.5,,...,,8.0,,10.5,,-4.2,,-999,,EOR
4,13711,20031112,-999.0,,-0.0,,-999.0,,-999.0,,...,,7.7,,10.5,,-999.0,,-999,,EOR


In [3]:
# Dátum → év (EZ HIÁNYZOTT)
df["Time"] = pd.to_datetime(df["Time"], format="%Y%m%d", errors="coerce")
df["Év"] = df["Time"].dt.year

# Hőmérséklet tisztítása
df["t"] = pd.to_numeric(df["t"], errors="coerce").replace(-999, None)

# Éves átlag
eves = df.groupby("Év")["t"].mean().reset_index()

# Grafikon
px.line(eves, x="Év", y="t", markers=True,
        title="Éves átlaghőmérséklet (°C)").show()


In [4]:
df["p"] = pd.to_numeric(df["p"], errors="coerce")

eves_atlag_legny = df.groupby("Év")["p"].mean().reset_index()

fig = px.line(
    eves_atlag_legny,
    x="Év",
    y="p",
    markers=True,
    title="Éves átlagos légnyomás (hPa)"
)

fig.show()


In [5]:
df["fs"] = pd.to_numeric(df["fs"], errors="coerce")

eves_atlag_szel = df.groupby("Év")["fs"].mean().reset_index()

fig = px.line(
    eves_atlag_szel,
    x="Év",
    y="fs",
    markers=True,
    title="Éves átlagos szélsebesség (m/s)"
)

fig.show()


In [6]:
# Hónap oszlop létrehozása
df["Hónap"] = df["Time"].dt.month

# Hónapnevek
honapnevek = {
    1: "január", 2: "február", 3: "március", 4: "április",
    5: "május", 6: "június", 7: "július", 8: "augusztus",
    9: "szeptember", 10: "október", 11: "november", 12: "december"
}

df["Hónap_név"] = df["Hónap"].map(honapnevek)

fig = px.box(
    df,
    x="Hónap_név",
    y="t",
    category_orders={"Hónap_név": [
        "január", "február", "március", "április", "május", "június",
        "július", "augusztus", "szeptember", "október", "november", "december"
    ]},
    title="Havi hőmérséklet-eloszlások"
)

fig.show()


In [7]:
valtozo_dropdown = widgets.Dropdown(
    options=[
        ("Hőmérséklet (t)", "t"),
        ("Szélsebesség (fs)", "fs"),
        ("Légnyomás (p)", "p")
    ],
    value="t",
    description="Változó:"
)

ev_slider = widgets.IntRangeSlider(
    value=[int(df["Év"].min()), int(df["Év"].max())],
    min=int(df["Év"].min()),
    max=int(df["Év"].max()),
    step=1,
    description="Év tartomány:",
    continuous_update=False
)

frissites_gomb = widgets.Button(
    description="Frissítés",
    button_style="success"
)

display(valtozo_dropdown, ev_slider, frissites_gomb)


Dropdown(description='Változó:', options=(('Hőmérséklet (t)', 't'), ('Szélsebesség (fs)', 'fs'), ('Légnyomás (…

IntRangeSlider(value=(2003, 2025), continuous_update=False, description='Év tartomány:', max=2025, min=2003)

Button(button_style='success', description='Frissítés', style=ButtonStyle())

In [8]:
valtozo_nevek = {
    "t": "Hőmérséklet (°C)",
    "fs": "Szélsebesség (m/s)",
    "p": "Légnyomás (hPa)"
}

def rajzol():
    d = df[
        (df["Év"] >= ev_slider.value[0]) &
        (df["Év"] <= ev_slider.value[1])
    ]

    fig = px.line(
        d,
        x="Time",
        y=valtozo_dropdown.value,
        title=f"{valtozo_nevek[valtozo_dropdown.value]} idősor ({ev_slider.value[0]}–{ev_slider.value[1]})",
        markers=True
    )
    fig.show()

rajzol()


In [13]:
import pandas as pd
import plotly.express as px

# --- 1) CSV beolvasása ---
df = pd.read_csv(
    "adatok_meteorologia.csv",
    encoding="latin2",
    sep=";",
    skiprows=5,
    engine="python",
    on_bad_lines="skip"
)

# --- 2) Oszlopnevek tisztítása ---
df.columns = df.columns.str.strip()

# --- 3) Dátum konvertálása ---
df["Time"] = pd.to_datetime(df["Time"], format="%Y%m%d")

# --- 4) Év oszlop ---
df["Év"] = df["Time"].dt.year

# --- 5) Választható változók ---
valtozok = {
    "Hőmérséklet (t)": "t",
    "Szélsebesség (fs)": "fs",
    "Légnyomás (p)": "p"
}

# --- 6) Dropdown menü létrehozása ---
fig = px.line(
    df,
    x="Time",
    y="t",
    title="Meteorológiai idősor – változó szerint"
)

fig.update_layout(
    updatemenus=[
        {
            "buttons": [
                {
                    "label": nev,
                    "method": "update",
                    "args": [
                        {"y": [df[oszlop]]},
                        {"title": f"{nev} idősora"}
                    ]
                }
                for nev, oszlop in valtozok.items()
            ],
            "direction": "down",
            "showactive": True,
            "x": 1.15,
            "y": 1.0
        }
    ],
    template="plotly_white",
    xaxis_title="Dátum",
    yaxis_title="Érték"
)

fig.show()


In [10]:
def frissites_gomb_kattintas(b):
    rajzol()

frissites_gomb.on_click(frissites_gomb_kattintas)



In [11]:
import pandas as pd
import plotly.express as px

# --- Adatbeolvasás ---
df = pd.read_csv(
    "adatok_meteorologia.csv",
    encoding="latin2",
    sep=";",
    skiprows=5,
    engine="python",
    on_bad_lines="skip"
)

# --- Oszlopnevek tisztítása ---
df.columns = df.columns.str.strip()

# --- Dátum konvertálása ---
df["Time"] = pd.to_datetime(df["Time"], format="%Y%m%d")
df["Év"] = df["Time"].dt.year

# --- HIBÁS HŐMÉRSÉKLET ÉRTÉKEK KISZŰRÉSE ---
df = df[df["t"] > -900]

# --- Választható változók ---
valtozok = {
    "Hőmérséklet (t)": "t",
    "Szélsebesség (fs)": "fs",
    "Légnyomás (p)": "p"
}

# --- Alap grafikon: SCATTER (pontgrafikon) ---
fig = px.scatter(
    df,
    x="Time",
    y=valtozok["Hőmérséklet (t)"],
    title="Meteorológiai idősor – változó szerint",
    template="plotly_white"
)

# --- Dropdown menü ---
fig.update_layout(
    updatemenus=[
        {
            "buttons": [
                {
                    "label": nev,
                    "method": "update",
                    "args": [
                        {"y": [df[oszlop]]},
                        {"title": f"{nev} idősora"}
                    ]
                }
                for nev, oszlop in valtozok.items()
            ],
            "direction": "down",
            "showactive": True,
            "x": 1.15,
            "y": 1.0
        }
    ],
    xaxis_title="Dátum",
    yaxis_title="Érték"
)

fig.show()


In [12]:
import pandas as pd
import plotly.express as px

# --- Adatbeolvasás ---
df = pd.read_csv(
    "adatok_meteorologia.csv",
    encoding="latin2",
    sep=";",
    skiprows=5,
    engine="python",
    on_bad_lines="skip"
)

# --- Oszlopnevek tisztítása ---
df.columns = df.columns.str.strip()

# --- Dátum konvertálása ---
df["Time"] = pd.to_datetime(df["Time"], format="%Y%m%d")
df["Év"] = df["Time"].dt.year

# --- Hibás értékek kiszűrése ---
df = df[df["t"] > -900]

# --- Választható változók ---
valtozok = {
    "Hőmérséklet (t)": "t",
    "Szélsebesség (fs)": "fs",
    "Légnyomás (p)": "p"
}

# --- Alap pontgrafikon (lila) ---
fig = px.scatter(
    df,
    x="Time",
    y="t",
    title="Hőmérséklet (t) – idősor",
    template="plotly_white",
    color_discrete_sequence=["purple"]
)

# --- Dropdown menü ---
fig.update_layout(
    updatemenus=[
        {
            "buttons": [
                {
                    "label": nev,
                    "method": "update",
                    "args": [
                        {"y": [df[oszlop]]},   # <-- csak az y változik
                        {
                            "title": f"{nev} – idősor",
                            "yaxis": {"title": nev},
                        }
                    ]
                }
                for nev, oszlop in valtozok.items()
            ],
            "direction": "down",
            "showactive": True,
            "x": 1.15,
            "y": 1.0
        }
    ],
    xaxis_title="Dátum",
    yaxis_title="Érték"
)

fig.show()



In [14]:
station_lat = 47.7147
station_lon = 16.6658

fig = px.scatter_mapbox(
    lat=[station_lat],
    lon=[station_lon],
    hover_name=["Fertőrákos"],
    zoom=7,
    title="Mérőállomás térképen"
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()


In [15]:
station_lat = 47.7147
station_lon = 16.6658

fig = px.scatter_mapbox(
    lat=[station_lat],
    lon=[station_lon],
    hover_name=["Fertőrákos"],
    zoom=7,
    title="Mérőállomás térképen"
)

fig.update_layout(
    mapbox=dict(
        style="white-bg",
        center=dict(lat=station_lat, lon=station_lon),
        zoom=7,
        layers=[
            dict(
                sourcetype="raster",
                source=[
                    "https://services.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}"
                ],
                below="traces"   # <<< EZ A LÉNYEG
            )
        ]
    ),
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()



In [16]:
# ============================================
# 0) Könyvtárak
# ============================================
import pandas as pd
import numpy as np
import plotly.express as px

# ============================================
# 1) Adatbeolvasás
# ============================================

df = pd.read_csv(
    "adatok_meteorologia.csv",
    encoding="latin2",
    sep=";",
    skiprows=5,
    engine="python",
    on_bad_lines="skip"
)

# ============================================
# 2) Oszlopnevek tisztítása
# ============================================

df.columns = df.columns.str.strip()

# ============================================
# 3) Dátum konvertálása + Év oszlop
# ============================================

df["Time"] = pd.to_datetime(df["Time"], format="%Y%m%d", errors="coerce")
df["Év"] = df["Time"].dt.year

# ============================================
# 4) Hibás értékek (-999) kiszűrése
# ============================================

df = df.replace([-999, -999.0], np.nan)

# ============================================
# 5) Választható változók
# ============================================

valtozok = {
    "Hőmérséklet (t)": "t",
    "Szélsebesség (fs)": "fs",
    "Légnyomás (p)": "p"
}

# ============================================
# 6) Alap lila pontgrafikon
# ============================================

fig = px.scatter(
    df,
    x="Time",
    y="t",
    title="Hőmérséklet (t) – idősor",
    template="plotly_white",
    color_discrete_sequence=["purple"],
    opacity=0.7
)

# ============================================
# 7) Dropdown menü – változóváltás
# ============================================

fig.update_layout(
    updatemenus=[
        {
            "buttons": [
                {
                    "label": nev,
                    "method": "update",
                    "args": [
                        {"y": [df[oszlop]]},
                        {
                            "title": f"{nev} – idősor",
                            "yaxis": {"title": nev},
                        }
                    ]
                }
                for nev, oszlop in valtozok.items()
            ],
            "direction": "down",
            "showactive": True,
            "x": 1.15,
            "y": 1.0
        }
    ],
    xaxis_title="Dátum",
    yaxis_title="Érték"
)

fig.show()


In [ ]:
print("""
PPKE – ITK
Script nyelvek / Tárgykód: P‑ITMIA‑0009

Meteorológiai idősorok feldolgozása Pythonban
hiányzó adatok, rugalmas beolvasás és interaktív megjelenítés

Készítette:
Hartai Mónika Valentina
levelezős hallgató
Mesterséges Intelligencia Alkalmazásai szakirányú továbbképzés
""")


PPKE – ITK
Script nyelvek / Tárgykód: P‑ITMIA‑0009

Meteorológiai idősorok feldolgozása Pythonban
hiányzó adatok, rugalmas beolvasás és interaktív megjelenítés

Készítette:
Hartai Mónika Valentina
levelezős hallgató
Mesterséges Intelligencia Alkalmazásai szakirányú továbbképzés



In [ ]:
print("""
Tartalom

1. Bevezetés
2. Az adatforrás bemutatása
3. Adatbeolvasás és hibakezelés
4. Időbeli bontás és aggregáció
5. Megjelenítés – grafikonok
   5.1. Éves átlaghőmérséklet – vonaldiagram
   5.2. Éves átlagos légnyomás – vonaldiagram
   5.3. Éves átlagos szélsebesség – vonaldiagram
   5.4. Havi átlaghőmérséklet – oszlopdiagram
6. Boxplot és havi eloszlások
7. Interaktív megjelenítés – a felhasználó szerepe
8. Sikeres és sikertelen utak, promptolás
9. Összegzés
10. Szélsebesség (fs) pontdiagram – PNG mentéssel
11. Teljes interaktív dashboard kód
12. Képernyőképek a Colabban generált grafikonokról
""")


Tartalom

1. Bevezetés
2. Az adatforrás bemutatása
3. Adatbeolvasás és hibakezelés
4. Időbeli bontás és aggregáció
5. Megjelenítés – grafikonok
   5.1. Éves átlaghőmérséklet – vonaldiagram
   5.2. Éves átlagos légnyomás – vonaldiagram
   5.3. Éves átlagos szélsebesség – vonaldiagram
   5.4. Havi átlaghőmérséklet – oszlopdiagram
6. Boxplot és havi eloszlások
7. Interaktív megjelenítés – a felhasználó szerepe
8. Sikeres és sikertelen utak, promptolás
9. Összegzés
10. Szélsebesség (fs) pontdiagram – PNG mentéssel
11. Teljes interaktív dashboard kód
12. Képernyőképek a Colabban generált grafikonokról



In [ ]:
print("""
1. Bevezetés

Ebben a projektmunkában egy nagyméretű, valós meteorológiai adathalmazt dolgozok fel Python nyelv segítségével.
A választott adatforrás egy több százezer sort tartalmazó CSV-fájl (adatok meterológia.csv), amely egy meteorológiai
állomás méréseinek több éves adatait tartalmazza. Az adatok időbélyeggel, hőmérséklet-, szélsebesség- és
légnyomás-értékekkel rendelkeznek.

A feladat célja az volt, hogy olyan megoldást készítsek, amely:
- rugalmasan olvassa be az adatokat,
- kezeli a hibás és hiányzó értékeket,
- vizuálisan is bemutatja az adatok főbb mintázatait,
- és lehetőséget ad a megjelenítés befolyásolására (szűrés, változóválasztás) anélkül, hogy magát az adatot módosítanám.

Különösen fontos volt számomra, hogy a „nyers” adatok mögött milyen történet rajzolódik ki; hogyan változik az évek során
a hőmérséklet, mennyire megbízhatóak a mérések, és mit kezdünk azzal, ha az adatok „nem tökéletesek”.
""")


1. Bevezetés

Ebben a projektmunkában egy nagyméretű, valós meteorológiai adathalmazt dolgozok fel Python nyelv segítségével.
A választott adatforrás egy több százezer sort tartalmazó CSV-fájl (adatok meterológia.csv), amely egy meteorológiai
állomás méréseinek több éves adatait tartalmazza. Az adatok időbélyeggel, hőmérséklet-, szélsebesség- és
légnyomás-értékekkel rendelkeznek.

A feladat célja az volt, hogy olyan megoldást készítsek, amely:
- rugalmasan olvassa be az adatokat,
- kezeli a hibás és hiányzó értékeket,
- vizuálisan is bemutatja az adatok főbb mintázatait,
- és lehetőséget ad a megjelenítés befolyásolására (szűrés, változóválasztás) anélkül, hogy magát az adatot módosítanám.

Különösen fontos volt számomra, hogy a „nyers” adatok mögött milyen történet rajzolódik ki; hogyan változik az évek során
a hőmérséklet, mennyire megbízhatóak a mérések, és mit kezdünk azzal, ha az adatok „nem tökéletesek”.



In [ ]:
print("""
2. Az adatforrás bemutatása

A feldolgozott fájl neve: adatok meterológia.csv.

A fájl jellemzői:
- mérete több mint 2,7 MB,
- több százezer sor,
- a fejléc a 6. sorban található,
- az oszlopok pontosvesszővel (;) vannak elválasztva,
- a szélsebesség oszlopban a hiányzó értékeket a -999 érték jelöli.

A legfontosabb oszlopok:
- Time – időbélyeg
- t – hőmérséklet (°C)
- fs – szélsebesség (m/s)
- p – légnyomás (hPa)

Az adathalmaz komplexitását egyrészt a nagy méret, másrészt a hibás és hiányzó adatok jelenléte adja.
Ez utóbbi különösen fontos, mert a hiányzó értékek kezelésének módja alapvetően befolyásolja az eredmények értelmezését.
""")


2. Az adatforrás bemutatása

A feldolgozott fájl neve: adatok meterológia.csv.

A fájl jellemzői:
- mérete több mint 2,7 MB,
- több százezer sor,
- a fejléc a 6. sorban található,
- az oszlopok pontosvesszővel (;) vannak elválasztva,
- a szélsebesség oszlopban a hiányzó értékeket a -999 érték jelöli.

A legfontosabb oszlopok:
- Time – időbélyeg
- t – hőmérséklet (°C)
- fs – szélsebesség (m/s)
- p – légnyomás (hPa)

Az adathalmaz komplexitását egyrészt a nagy méret, másrészt a hibás és hiányzó adatok jelenléte adja.
Ez utóbbi különösen fontos, mert a hiányzó értékek kezelésének módja alapvetően befolyásolja az eredmények értelmezését.



In [ ]:
print("""
3. Adatbeolvasás és hibakezelés

A projekt egyik kulcspontja a rugalmas adatbeolvasás volt. A CSV elején metaadat-blokkok találhatók, ezért a fejléc nem
az első sorban van. A beolvasás fő lépései:

- a fejléc sorának manuális azonosítása (header=5),
- hibás sorok átugrása (on_bad_lines="skip"),
- a -999 értékek NaN-ná alakítása,
- a dátumok konvertálása (errors="coerce").

A szélsebesség esetében külön probléma volt, hogy a hiányzó értékeket -999 jelölte. Első futtatáskor az éves átlagos
szélsebesség minden évben -999 lett, ami nyilvánvalóan hibás. Ez mutatta meg, hogy a hiányzó adatok felismerése és
kezelése nem pusztán technikai részlet, hanem az értelmezés szempontjából is döntő.


4. Időbeli bontás és aggregáció

A Time oszlopból két új oszlopot hoztam létre:
- Év
- Hónap

Ez lehetővé tette:
- éves átlagok számítását (t, fs, p),
- havi átlagok számítását (t),
- havi eloszlások (boxplot) készítését.


5. Megjelenítés – grafikonok

5.1. Éves átlaghőmérséklet – vonaldiagram
A hőmérséklet éves átlagait vonaldiagramon ábrázoltam. A grafikon jól mutatja az évek közötti ingadozásokat és a hosszú távú trendeket.

5.2. Éves átlagos légnyomás – vonaldiagram
A légnyomás éves átlagai kisebb ingadozásokat mutatnak, ami meteorológiai szempontból természetes.

5.3. Éves átlagos szélsebesség – vonaldiagram
A szélsebesség esetében a hiányzó adatok miatt a grafikon inkább azt mutatja meg, hogy mely években áll rendelkezésre
értelmezhető mennyiségű adat. A -999 értékeket NaN-ná alakítottam, így az éves átlagok csak a valós mérésekből számolódnak.

5.4. Havi átlaghőmérséklet – oszlopdiagram
A havi átlaghőmérsékletek szépen kirajzolják az évszakok ritmusát.


6. Boxplot és havi eloszlások

A boxplot megmutatja a hőmérséklet eloszlását hónaponként, beleértve a mediánt, szórást és szélsőértékeket.


7. Interaktív megjelenítés – a felhasználó szerepe

A felhasználó:
- kiválaszthatja a változót (t, fs, p),
- beállíthatja az év tartományt,
- interaktív idősorokat és éves átlagokat láthat,
- térképen megjelenik a Fertőrákos állomás.

Ez a projekt egyik legfontosabb része, mert a megjelenítés befolyásolható anélkül, hogy az adatot módosítanánk.


8. Sikeres és sikertelen utak, promptolás

- A szélsebesség átlagolása elsőre hibás volt (minden év -999).
- A fejléc nem az első sorban volt → rugalmas beolvasás kellett.
- A promptolás segített a hibák felismerésében és javításában.


9. Összegzés

A projektmunka során sikerült:
- rugalmas adatbeolvasást készíteni,
- a hiányzó adatokat megfelelően kezelni,
- többféle statikus és interaktív grafikont készíteni,
- a felhasználót bevonni a vizualizációba.

A projekt teljesíti a Script nyelvek tárgy követelményeit.
""")


3. Adatbeolvasás és hibakezelés

A projekt egyik kulcspontja a rugalmas adatbeolvasás volt. A CSV elején metaadat-blokkok találhatók, ezért a fejléc nem
az első sorban van. A beolvasás fő lépései:

- a fejléc sorának manuális azonosítása (header=5),
- hibás sorok átugrása (on_bad_lines="skip"),
- a -999 értékek NaN-ná alakítása,
- a dátumok konvertálása (errors="coerce").

A szélsebesség esetében külön probléma volt, hogy a hiányzó értékeket -999 jelölte. Első futtatáskor az éves átlagos
szélsebesség minden évben -999 lett, ami nyilvánvalóan hibás. Ez mutatta meg, hogy a hiányzó adatok felismerése és
kezelése nem pusztán technikai részlet, hanem az értelmezés szempontjából is döntő.


4. Időbeli bontás és aggregáció

A Time oszlopból két új oszlopot hoztam létre:
- Év
- Hónap

Ez lehetővé tette:
- éves átlagok számítását (t, fs, p),
- havi átlagok számítását (t),
- havi eloszlások (boxplot) készítését.


5. Megjelenítés – grafikonok

5.1. Éves átlaghőmérséklet – vonaldiagram
A hőmérsé

In [ ]:
print("""
12. Képernyőképek a Colabban generált grafikonokról

A beadandóhoz tartozó képernyőképek (PNG fájlok) ide kerülnek:

- eves_atlaghomerseklet.png
- eves_atlag_legnyomas.png
- eves_atlag_szel.png
- havi_atlaghomerseklet.png
- boxplot_havi_homerseklet.png
- szelsebesseg_fs.png
- dashboard_kepernyokepek

A PNG fájlokat a Colab bal oldali 'Files' paneljén lehet feltölteni vagy letölteni.
""")


12. Képernyőképek a Colabban generált grafikonokról

A beadandóhoz tartozó képernyőképek (PNG fájlok) ide kerülnek:

- eves_atlaghomerseklet.png
- eves_atlag_legnyomas.png
- eves_atlag_szel.png
- havi_atlaghomerseklet.png
- boxplot_havi_homerseklet.png
- szelsebesseg_fs.png
- dashboard_kepernyokepek

A PNG fájlokat a Colab bal oldali 'Files' paneljén lehet feltölteni vagy letölteni.



In [ ]:
print("""
Tisztelt Dr. Tornai Kálmán Dékánhelyettes, Egyetemi Docens Úr!

A projektmunka interaktív elemei (grafikonok, csúszkák, térképes megjelenítés) a Google Colab környezetben
futtathatók. A jegyzetfüzet teljes tartalmának megjelenítéséhez az alábbi lépések követése szükséges:

1. A notebook megnyitása után kérem válassza a felső menüsorban a
   'Runtime' / 'Futtatás' menüpontot.

2. Ezen belül kérem indítsa el az 'Run all' / 'Összes futtatása' parancsot.

3. A futtatás néhány másodpercet igénybe vehet. A cellák egymás után lefutnak,
   és megjelennek:
   - a statikus grafikonok,
   - a szélsebesség pontdiagram,
   - valamint az interaktív dashboard, amelyben a változó és az év tartomány
     szabadon állítható.

4. Az interaktív elemek (Plotly grafikonok, csúszkák, térkép) a futtatás után
   azonnal használhatók.

Köszönöm szépen a beadandó megtekintését és értékelését!

Tisztelettel:
Hartai Mónika Valentina
""")


Tisztelt Dr. Tornai Kálmán Dékánhelyettes, Egyetemi Docens Úr!

A projektmunka interaktív elemei (grafikonok, csúszkák, térképes megjelenítés) a Google Colab környezetben
futtathatók. A jegyzetfüzet teljes tartalmának megjelenítéséhez az alábbi lépések követése szükséges:

1. A notebook megnyitása után kérem válassza a felső menüsorban a
   'Runtime' / 'Futtatás' menüpontot.

2. Ezen belül kérem indítsa el az 'Run all' / 'Összes futtatása' parancsot.

3. A futtatás néhány másodpercet igénybe vehet. A cellák egymás után lefutnak,
   és megjelennek:
   - a statikus grafikonok,
   - a szélsebesség pontdiagram,
   - valamint az interaktív dashboard, amelyben a változó és az év tartomány
     szabadon állítható.

4. Az interaktív elemek (Plotly grafikonok, csúszkák, térkép) a futtatás után
   azonnal használhatók.

Köszönöm szépen a beadandó megtekintését és értékelését!

Tisztelettel:
Hartai Mónika Valentina



In [ ]:
print("A notebook futtatása során létrejött fájlok listája:\n")

import os

files = os.listdir(".")   # Jelenlegi könyvtár
for f in files:
    print("-", f)

A notebook futtatása során létrejött fájlok listája:

- .bashrc
- .bash_logout
- .profile
- Futtatandó meterológia projektmunka_Hartai Mónika V._PPKE Script nyelvek.ipynb
- homerseklet_teljes_idosor.html
- .npm
- .local
- .ipython
- .jupyter-server-log.txt
- .ipynb_checkpoints
- .jupyter
- projektmunka.ipynb
- .cache
- .config
- szelsebesseg_fs.png
- README.md
- requirements.txt
- adatok meterológia.csv
- .git


In [ ]:
print("""
Tisztelt Dr. Tornai Kálmán Dékánhelyettes, Egyetemi Docens Úr!

Szeretném megköszönni a félév során kapott útmutatásokat és a lehetőséget,
hogy ezt a projektmunkát elkészíthettem. A feladat során sokat tanultam a
Python adatfeldolgozási és vizualizációs lehetőségeiről, valamint az
interaktív megjelenítés gyakorlati alkalmazásáról.

A notebookban szándékosan nem töröltem a futtatás során keletkezett
kiegészítő fájlokat (PNG, HTML stb.), hogy a teljes munkafolyamat
átlátható legyen, és a beadandó minden lépése, eredménye és melléklete
egy helyen megtekinthető maradjon.

Köszönöm szépen a beadandó megtekintését és értékelését!

Tisztelettel:
Hartai Mónika Valentina
""")


Tisztelt Dr. Tornai Kálmán Dékánhelyettes, Egyetemi Docens Úr!

Szeretném megköszönni a félév során kapott útmutatásokat és a lehetőséget,
hogy ezt a projektmunkát elkészíthettem. A feladat során sokat tanultam a
Python adatfeldolgozási és vizualizációs lehetőségeiről, valamint az
interaktív megjelenítés gyakorlati alkalmazásáról.

A notebookban szándékosan nem töröltem a futtatás során keletkezett
kiegészítő fájlokat (PNG, HTML stb.), hogy a teljes munkafolyamat
átlátható legyen, és a beadandó minden lépése, eredménye és melléklete
egy helyen megtekinthető maradjon.

Köszönöm szépen a beadandó megtekintését és értékelését!

Tisztelettel:
Hartai Mónika Valentina

